In [1]:
#import libraries 
from pathlib import Path
import torch

from tokenizers import Tokenizer

DATA_DIR = Path("/srv/data/lt2326-h26/a1")

Tokenizer_DIR = Path("tokenizer/tokenizers")

Languages = ["en", "tr", "zh"]

print("Notebook working directory:", Path.cwd())
print("Dataset directory exists:", DATA_DIR.exists())
print("Tokenizer directory exists:", Tokenizer_DIR.exists())

Notebook working directory: /home/gussulthsa@GU.GU.SE/multilingual-tokenization-lm/src
Dataset directory exists: True
Tokenizer directory exists: True


In [2]:


# Find the dataset files
for split in ["train", "valid", "test"]:
    split_dir = DATA_DIR / split
    print(f"\n{split.upper()}")

    if split_dir.exists():
        for path in sorted(split_dir.rglob("*")):
            if path.is_file():
                print(path)
    else:
        print(f"Directory not found: {split_dir}")

# Find your saved tokenizers
print("\nSAVED TOKENIZER FILES")
if Tokenizer_DIR.exists():
    for path in sorted(Tokenizer_DIR.rglob("*")):
        if path.is_file():
            print(path)
else:
    print("Tokenizer directory not found:", Tokenizer_DIR.resolve())


TRAIN
/srv/data/lt2326-h26/a1/train/en.txt
/srv/data/lt2326-h26/a1/train/tr.txt
/srv/data/lt2326-h26/a1/train/zh.txt

VALID
/srv/data/lt2326-h26/a1/valid/en.txt
/srv/data/lt2326-h26/a1/valid/tr.txt
/srv/data/lt2326-h26/a1/valid/zh.txt

TEST
/srv/data/lt2326-h26/a1/test/en.txt
/srv/data/lt2326-h26/a1/test/tr.txt
/srv/data/lt2326-h26/a1/test/zh.txt

SAVED TOKENIZER FILES
tokenizer/tokenizers/bpe_10k.json
tokenizer/tokenizers/bpe_2k.json


In [3]:
#Load a small sample from each training language

#read only first 2,000 characters from each file
for lang in Languages:
    path = DATA_DIR / "train" / f"{lang}.txt"
    with open(path, "r", encoding="utf-8") as f:
        text = f.read(2000)

    print(f"\\n{lang.upper()}:")
    print("Text sample:", repr(text[:150]))
    print("Characters read:", len(text))

\nEN:
Text sample: 'Since Jack Kirby had more than enough assignments, Lee assigned John Buscema the task of illustrating the new book."\nInuit means more than one, one pe'
Characters read: 2000
\nTR:
Text sample: "Ekonomi alanındaki bu gelişmelerin yanında, okuma-yazma bilmezliğe karşı yoğun bir savaşım görüldü.\nBabasının isteği üzerine 1864'te hukukçu olmak içi"
Characters read: 2000
\nZH:
Text sample: '他直到死前都沒有親自管理過公國。\n，面積205平方公里，海拔高度611米，2007年人口6,597，人口密度每平方公里32人。\n帝谓：“善长虽无汗马劳，然事朕久，给军食，功甚大，宜进封大国。”\n，解放军占领全部宁夏，战役结束 。\n任何一方：Termination（PADT）\n四条轨道中，中间两条为房'
Characters read: 2000


In [4]:
#Load the training and validation text
#Read the training data for all three languages into a dictionary
train_texts = {
    lang: (DATA_DIR / "train" / f"{lang}.txt").read_text(encoding="utf-8")
    for lang in Languages
}

#read the validation data for all three languages into a dictionary
valid_texts = {
    lang: (DATA_DIR / "valid" / f"{lang}.txt").read_text(encoding="utf-8")
    for lang in Languages
}

#check the number of characters in each training and validation text
print("Training characters:")
for lang in Languages:
    print(f"{lang.upper()}: {len(train_texts[lang]):,}")

print("\nValidation characters:")
for lang in Languages:
    print(f"{lang.upper()}: {len(valid_texts[lang]):,}")


Training characters:
EN: 2,982,219
TR: 2,976,519
ZH: 3,022,139

Validation characters:
EN: 372,809
TR: 372,118
ZH: 377,820


In [5]:
#Load the saved BPE tokenizers we trained earlier
bpe_2k = Tokenizer.from_file(str(Tokenizer_DIR / "bpe_2k.json"))
bpe_10k= Tokenizer.from_file(str(Tokenizer_DIR / "bpe_10k.json"))

print("BPE-2k vocabulary:", bpe_2k.get_vocab_size())
print("BPE-10k vocabulary:", bpe_10k.get_vocab_size())


BPE-2k vocabulary: 2000
BPE-10k vocabulary: 10000


In [6]:
#testing how BPE tokenizer split text in each language
sample_text = {
    "en": "Hello, how are you?",
    "tr": "Merhaba, nasılsın?",
    "zh": "你好，你怎么样？"
}

for lang, text in sample_text.items():
    print(f"\n {lang.upper()}: original text")
    print(text)

encoded_2k = bpe_2k.encode(text)
encoded_10k = bpe_10k.encode(text)

#show the tokens produced by each tokenizer
print("Length of BPE-2k tokens:", len(encoded_2k))
print("BPE-2k IDs:", encoded_2k.ids)
print("BPE-2k tokens:", encoded_2k.tokens)

print("Length of BPE-10k tokens:", len(encoded_10k))
print("BPE-10k IDS:", encoded_10k.ids)
print("BPE-10k tokens:", encoded_10k.tokens)


 EN: original text
Hello, how are you?

 TR: original text
Merhaba, nasılsın?

 ZH: original text
你好，你怎么样？
Length of BPE-2k tokens: 15
BPE-2k IDs: [221, 306, 255, 1265, 258, 306, 255, 478, 237, 321, 231, 473, 116, 257, 254]
BPE-2k tokens: ['Ġ', 'ä½', 'ł', 'å¥½', 'ï¼Į', 'ä½', 'ł', 'æĢ', 'İ', 'ä¹', 'Ī', 'æł', '·', 'ï¼', 'Ł']
Length of BPE-10k tokens: 10
BPE-10k IDS: [221, 3029, 1265, 258, 3029, 478, 237, 4494, 2225, 3964]
BPE-10k tokens: ['Ġ', 'ä½ł', 'å¥½', 'ï¼Į', 'ä½ł', 'æĢ', 'İ', 'ä¹Ī', 'æł·', 'ï¼Ł']


Encoding a small english sample and shift the targets

In [7]:
#Read 500 characters from the English training file
with open(DATA_DIR / "train" / "en.txt", "r", encoding="utf-8") as f:
    sample_text = f.read(500)

# Convert the sample text to BPE-2K token IDs.
sample_ids = bpe_2k.encode(sample_text).ids

print("Original text:")
print(repr(sample_text[:200]))
print("\\nNumber of characters:", len(sample_text))
print("Number of BPE tokens:", len(sample_ids))
print("First 30 token IDs:", sample_ids[:30])

# For next-token prediction, the target is the input shifted by one token.
input_ids = sample_ids[:-1]
target_ids = sample_ids[1:]

print("\\nFirst 10 input IDs: ", input_ids[:10])
print("First 10 target IDs:", target_ids[:10])
print("Number of input tokens:", len(input_ids))
print("Number of target tokens:", len(target_ids))

Original text:
'Since Jack Kirby had more than enough assignments, Lee assigned John Buscema the task of illustrating the new book."\nInuit means more than one, one person is an "Inuk".\nSunday edition until his death '
\nNumber of characters: 500
Number of BPE tokens: 226
First 30 token IDs: [364, 264, 656, 706, 1276, 476, 286, 66, 89, 1322, 1852, 456, 263, 960, 79, 1560, 673, 83, 1942, 1570, 83, 12, 603, 69, 69, 673, 83, 1942, 294, 706]
\nFirst 10 input IDs:  [364, 264, 656, 706, 1276, 476, 286, 66, 89, 1322]
First 10 target IDs: [264, 656, 706, 1276, 476, 286, 66, 89, 1322, 1852]
Number of input tokens: 225
Number of target tokens: 225


Building the character vocabulary from traing data

In [8]:

#combining the training text from all three languages

all_train_text = "".join(train_texts[lang] for lang in Languages)

#finding all unique characters in the training text
characters = sorted(set(all_train_text))

#Assign an integer ID to each character
#Id 0 is reserved for unknown characters
stoi = {"<UNK>": 0}

for i, char in enumerate(characters, start=1):
    stoi[char] = i

#Create the reverse mapping from IDs back to characters
itos = {i: char for char, i in stoi.items()}

#convert text into character IDs
#if char not in our vocabulary, use the unknown ID
def encode_char(text):
    return [stoi.get(char, stoi["<UNK>"]) for char in text]

print("Character vocabulary size:", len(stoi))
print("Newline token ID:", stoi.get("\n"))
print("Unknown token Id:", stoi["<UNK>"])


Character vocabulary size: 9444
Newline token ID: 1
Unknown token Id: 0


Define the tokenizers and count training tokens

In [9]:
#Each function converts text into a list of token IDs
tokenizers = {
    "character": encode_char,
    "bpe_2k": lambda text: bpe_2k.encode(text).ids,
    "bpe_10k": lambda text: bpe_10k.encode(text).ids
}
#store vocabulary sizes for the three model configurations
vocab_sizes = {
    "character": len(stoi),
    "bpe_2k": bpe_2k.get_vocab_size(),
    "bpe_10k": bpe_10k.get_vocab_size(),
}
# Count the tokens produced from each training file
print("Training token counts:")
for tokenizer_name, encode_fn in tokenizers.items():
    print(f"\n{tokenizer_name}")

    for lang in Languages:
        ids = encode_fn(train_texts[lang])
        print(f"{lang.upper()}: {len(ids):,} tokens")

print("\\nVocabulary sizes:")
for tokenizer_name, vocab_size in vocab_sizes.items():
    print(f"{tokenizer_name}: {vocab_size:,}")

Training token counts:

character
EN: 2,982,219 tokens
TR: 2,976,519 tokens
ZH: 3,022,139 tokens

bpe_2k
EN: 1,363,691 tokens
TR: 1,380,794 tokens
ZH: 3,357,246 tokens

bpe_10k
EN: 968,065 tokens
TR: 948,949 tokens
ZH: 2,428,594 tokens
\nVocabulary sizes:
character: 9,444
bpe_2k: 2,000
bpe_10k: 10,000


#converting the training text into tensors

In [10]:

# Store the token IDs for each language and tokenizer
# Each language has its own sequence of token IDs
train_data = {
    "character": {},
    "bpe_2k": {},
    "bpe_10k": {}
}

for tokenizer_name, encode_fn in tokenizers.items():
    for lang in Languages:

        # Convert the text into token IDs
        token_ids = encode_fn(train_texts[lang])

        # Convert the list of IDs into a PyTorch tensor
        # We use long because the model's embedding layer expects integer IDs
        train_data[tokenizer_name][lang] = torch.tensor(
            token_ids, dtype=torch.long
        )

# Check the size of each tensor
print("Training tensor shapes and token counts:")

for tokenizer_name, language_data in train_data.items():
    print(f"\n{tokenizer_name}")

    for lang, tensor in language_data.items():
        print(
            f"{lang.upper()}: shape={tuple(tensor.shape)},"
             f"tokens={len(tensor):,}"
             )

Training tensor shapes and token counts:

character
EN: shape=(2982219,),tokens=2,982,219
TR: shape=(2976519,),tokens=2,976,519
ZH: shape=(3022139,),tokens=3,022,139

bpe_2k
EN: shape=(1363691,),tokens=1,363,691
TR: shape=(1380794,),tokens=1,380,794
ZH: shape=(3357246,),tokens=3,357,246

bpe_10k
EN: shape=(968065,),tokens=968,065
TR: shape=(948949,),tokens=948,949
ZH: shape=(2428594,),tokens=2,428,594


Creating batches for next token prediction

In [11]:
#creating a batch of input and targer sequences
import random

# Settings for our training batches
context_length = 128
batch_size = 16

# Use the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)


def get_batch(tokenizer_name, batch_size=16, context_length=128):

    # Get the training data for the selected tokenizer
    language_data = train_data[tokenizer_name]
    language_names = list(language_data.keys())

    inputs = []
    targets = []

    for _ in range(batch_size):

        # Randomly choose one of the three languages
        lang = random.choice(language_names)
        data = language_data[lang]

        # Choose a random starting position in the text
        max_start = len(data) - context_length - 1
        start = torch.randint(0, max_start + 1, (1,)).item()

        # Input contains 128 consecutive tokens
        input_sequence = data[start:start + context_length]

        # Target contains the next token at each position
        target_sequence = data[start + 1:start + context_length + 1]

        inputs.append(input_sequence)
        targets.append(target_sequence)

    # Combine the sequences into batches and move them to the GPU
    inputs = torch.stack(inputs).to(device)
    targets = torch.stack(targets).to(device)

    return inputs, targets

Training device: cuda


In [12]:
#Testing the batch function
# Check that our batch function works for all three tokenizers
for tokenizer_name in train_data:

    inputs, targets = get_batch(
        tokenizer_name,
        batch_size=batch_size,
        context_length= context_length,
        )

    print(f"\n{tokenizer_name}")
    print("Input shape:", inputs.shape)
    print("Target shape:", targets.shape)
    print("Input device:", inputs.device)

    # The target sequence should be shifted one position
    # compared with the input sequence
    print(
        "Shift check:",
        torch.equal(inputs[:, 1:], targets[:, :-1])
    )


character
Input shape: torch.Size([16, 128])
Target shape: torch.Size([16, 128])
Input device: cuda:0
Shift check: True

bpe_2k
Input shape: torch.Size([16, 128])
Target shape: torch.Size([16, 128])
Input device: cuda:0
Shift check: True

bpe_10k
Input shape: torch.Size([16, 128])
Target shape: torch.Size([16, 128])
Input device: cuda:0
Shift check: True


In [13]:

# Check how the BPE tokenizers process Chinese text

print("BPE-2K pre-tokenizer:")
print(bpe_2k.pre_tokenizer)

print("\nBPE-2K decoder:")
print(bpe_2k.decoder)

print("\nBPE-10K pre-tokenizer:")
print(bpe_10k.pre_tokenizer)

print("\nBPE-10K decoder:")
print(bpe_10k.decoder)

# Decode the Chinese example to check that the original text is recovered
chinese_text = "你好，你怎么样？"

for name, tokenizer in [
    ("BPE-2K", bpe_2k),
    ("BPE-10K", bpe_10k),
]:
    encoded = tokenizer.encode(chinese_text)
    decoded = tokenizer.decode(encoded.ids)

    print(f"\n{name}")
    print("Original:", repr(chinese_text))
    print("Tokens:", encoded.tokens)
    print("Decoded:", repr(decoded))

BPE-2K pre-tokenizer:
ByteLevel(add_prefix_space=True, trim_offsets=True, use_regex=True)

BPE-2K decoder:
ByteLevel(add_prefix_space=True, trim_offsets=True, use_regex=True)

BPE-10K pre-tokenizer:
ByteLevel(add_prefix_space=True, trim_offsets=True, use_regex=True)

BPE-10K decoder:
ByteLevel(add_prefix_space=True, trim_offsets=True, use_regex=True)

BPE-2K
Original: '你好，你怎么样？'
Tokens: ['Ġ', 'ä½', 'ł', 'å¥½', 'ï¼Į', 'ä½', 'ł', 'æĢ', 'İ', 'ä¹', 'Ī', 'æł', '·', 'ï¼', 'Ł']
Decoded: ' 你好，你怎么样？'

BPE-10K
Original: '你好，你怎么样？'
Tokens: ['Ġ', 'ä½ł', 'å¥½', 'ï¼Į', 'ä½ł', 'æĢ', 'İ', 'ä¹Ī', 'æł·', 'ï¼Ł']
Decoded: ' 你好，你怎么样？'
